In [ ]:
"""Blend file41 (6.576 real LB, older 0.594-lineage base + beam2/neighbor/WARP) with file44
(6.464 real LB, newer 0.462-lineage base + beam2/neighbor/WARP+Q0522 hedge). NOTE: unlike the
p21+p44 blend, these two are NOT independent pipelines -- diff check showed they agree exactly
on 2/3 real test wells and differ only on 00e12e8b (mean 0.35ft), which is precisely the well the
competitor's Q0522 cell hand-tunes against the public score. This blend is really a half-dose of
that specific hedge, not a cross-lineage ensemble -- testing whether the hedge is LB-overfit.
Uses TWO separate single-file datasets (not one combined dataset) -- user's hypothesis is that
Kaggle's submission-scoring pipeline mis-handles kernels with only a single non-competition
dataset attached, which matches both prior exceptions (file45, file50 v1) happening only on
single-dataset blend notebooks and never on the multi-dataset heavy pipelines.
"""
import glob, os, time, pandas as pd

BLEND_W = 0.50  # weight on p44; (1-BLEND_W) on p41

def _wait_for(pattern, tries=10, delay=15):
    for i in range(tries):
        hits = glob.glob(pattern, recursive=True)
        if hits:
            if i > 0:
                print(f'{pattern}: found after {i} retries', flush=True)
            return hits
        print(f'{pattern}: not found yet (try {i+1}/{tries}), waiting {delay}s', flush=True)
        time.sleep(delay)
    return []

_p41_hits = _wait_for('/kaggle/input/**/p41_submission.csv')
assert _p41_hits, 'p41_submission.csv not found -- attach the rogii-p41-submission-only dataset'
_p44_hits = _wait_for('/kaggle/input/**/p44_submission.csv')
assert _p44_hits, 'p44_submission.csv not found -- attach the rogii-p44-submission-only dataset'

_comp = _wait_for('/kaggle/input/**/sample_submission.csv')
assert _comp, 'sample_submission.csv not found -- attach the competition dataset'
SAMPLE = _comp[0]

p41 = pd.read_csv(_p41_hits[0], dtype={'id': 'string'})
p44 = pd.read_csv(_p44_hits[0], dtype={'id': 'string'})
sample = pd.read_csv(SAMPLE, dtype={'id': 'string'})[['id']]

p41 = sample.merge(p41, on='id', how='left')
p44 = sample.merge(p44, on='id', how='left')
assert p41['tvt'].notna().all() and p44['tvt'].notna().all(), 'missing ids after merge'
assert (p41['id'] == p44['id']).all() and (p41['id'] == sample['id']).all()

blended = sample.copy()
blended['tvt'] = (1.0 - BLEND_W) * p41['tvt'].to_numpy(float) + BLEND_W * p44['tvt'].to_numpy(float)

assert len(blended) == len(sample)
assert blended['tvt'].notna().all()
blended[['id', 'tvt']].to_csv('submission.csv', index=False)
print(f'blend_w(p44)={BLEND_W}  rows={len(blended)}  '
      f'tvt mean={blended["tvt"].mean():.3f} std={blended["tvt"].std():.3f}')
print('saved submission.csv')